# Use Case: Evaluating Online TV Advertisements with a Simulated Audience

This notebook demonstrates a practical use case for TinyTroupe: evaluating the effectiveness of different online advertisement copies by simulating how a diverse group of agents (potential customers) would react to them.

The process involves:
1. Defining several distinct advertisement texts for a product (in this case, TVs).
2. Crafting a prompt that asks agents to choose the most convincing ad based on their persona.
3. Creating a relevant situation or context for the agents (e.g., their current TV is broken).
4. First, using a pre-defined agent to test the evaluation process.
5. Then, generating a diverse panel of new agents using `TinyPersonFactory` to represent a varied audience.
6. Having each agent "review" the ads and state their preference.
7. Using `ResultsExtractor` to systematically pull out the chosen ad from each agent's response.
8. Aggregating the results to determine which ad was most persuasive across the simulated audience.

## 1. Setup and Imports

Import necessary modules: `TinyPerson` for agents, `TinyPersonFactory` to create diverse agents, and `ResultsExtractor` to parse their choices. `sys` is used to adjust the Python path for finding the `tinytroupe` library if running from the examples directory.

In [ ]:
import json
import sys
# If running from the 'examples' directory, this adds the parent directory (project root) to the Python path.
sys.path.insert(0, '../..') # Adjusted path to go up two levels to project root

import tinytroupe # Initializes configuration, logging, etc.
from tinytroupe.agent import TinyPerson
# Example helper to create a specific agent, Lisa.
from tinytroupe.examples import create_lisa_the_data_scientist 
from tinytroupe.factory import TinyPersonFactory
from tinytroupe.extraction import ResultsExtractor

## 2. Define Advertisement Copies

Here are three different ad copies for TVs. These were obtained from real Bing search queries for "55 inches tv".

In [ ]:
# User search query: "55 inches tv"

tv_ad_1 =\
"""
The Best TV Of Tomorrow - LG 4K Ultra HD TV
https://www.lg.com/tv/oled
AdThe Leading Name in Cinematic Picture. Upgrade Your TV to 4K OLED And See The Difference. It's Not Just OLED, It's LG OLED. Exclusive a9 Processor, Bringing Cinematic Picture Home.

Infinite Contrast · Self-Lighting OLED · Dolby Vision™ IQ · ThinQ AI w/ Magic Remote

Free Wall Mounting Deal
LG G2 97" OLED evo TV
Free TV Stand w/ Purchase
World's No.1 OLED TV
"""

tv_ad_2 =\
"""
The Full Samsung TV Lineup - Neo QLED, OLED, 4K, 8K & More
https://www.samsung.com
AdFrom 4K To 8K, QLED To OLED, Lifestyle TVs & More, Your Perfect TV Is In Our Lineup. Experience Unrivaled Technology & Design In Our Ultra-Premium 8K & 4K TVs.

Discover Samsung Event · Real Depth Enhancer · Anti-Reflection · 48 mo 0% APR Financing

The 2023 OLED TV Is Here
Samsung Neo QLED 4K TVs
Samsung Financing
Ranked #1 By The ACSI®
"""

tv_ad_3 =\
"""
Wayfair 55 Inch Tv - Wayfair 55 Inch Tv Décor
Shop Now
https://www.wayfair.com/furniture/free-shipping
AdFree Shipping on Orders Over $35. Shop Furniture, Home Décor, Cookware & More! Free Shipping on All Orders Over $35. Shop 55 Inch Tv, Home Décor, Cookware & More!
"""

## 3. Craft the Evaluation Request

We'll create a message that will be sent to each agent, asking them to evaluate the ads and pick the one that is most convincing to them, based on their persona (financial situation, background, personality).

In [ ]:
eval_request_msg = \
f"""
Can you evaluate these Bing ads for me? Which one convices you more to buy their particular offering? 
Select **ONLY** one. Please explain your reasoning, based on your financial situation, background and personality.

# AD 1
```
{tv_ad_1}
```

# AD 2
```
{tv_ad_2}
```

# AD 3
```
{tv_ad_3}
```
"""

print("Evaluation request preview:")
print(eval_request_msg[:500] + "...") # Print a preview

## 4. Define the Agent's Situation

To make the evaluation more realistic, we'll give the agents a reason why they are looking for a new TV.

In [ ]:
situation = "Your current TV just broke and you urgently need a new one. You are searching on Bing for a 55-inch TV and these are the ads you see."

## 5. Test with a Pre-defined Agent (Lisa)

Let's use Lisa, our data scientist agent, to test the ad evaluation process.

In [ ]:
# Create Lisa using an example helper function
lisa = create_lisa_the_data_scientist()
print(f"Using agent: {lisa.name}\n")

# Set Lisa's current context/situation
lisa.change_context([situation])

# Have Lisa listen to the evaluation request and act (i.e., respond)
print(f"Presenting ads to {lisa.name}...")
lisa.listen_and_act(eval_request_msg)

print(f"\n{lisa.name}'s response should be visible above in the interaction log.")

### 5.1. Extract Lisa's Choice

We use `ResultsExtractor` to parse Lisa's response and identify which ad she chose.

In [ ]:
extractor = ResultsExtractor()

extraction_objective="Find the ad the agent chose. Extract the Ad number and title. Extract only ONE result."

lisa_choice = extractor.extract_results_from_agent(
    person=lisa, 
    extraction_objective=extraction_objective,
    situation=situation, # Providing the situation can help the extractor understand context
    fields=["ad_number", "ad_title"], # Define the JSON fields to extract
    fields_hints={"ad_number": "Must be an integer (1, 2, or 3), not a string like 'AD 1'."}, # Guide the LLM for correct extraction
    verbose=True # Shows the raw result from the LLM before parsing
)

if lisa_choice:
    print(f"\n{lisa.name}'s choice: AD {lisa_choice.get('ad_number')} - {lisa_choice.get('ad_title')}")
else:
    print(f"\nCould not extract a definitive choice from {lisa.name}.")

## 6. Evaluate with a Diverse Panel of Generated Agents

To get a broader perspective, we'll generate a panel of 5 diverse agents using `TinyPersonFactory`. The factory will be given a context to guide the generation towards varied personas.

### 6.1. Create Agent Factory and Generate Agents

In [ ]:
# Define a context for the factory to generate diverse personas
factory_context = """
A diverse group of individuals from different walks of life, socioeconomic statuses, professions, and with varied interests. 
Some are tech-savvy, others are budget-conscious. Some prioritize brand reputation, others look for specific features.
Ensure a mix of financial situations: some with high disposable income, others on a tighter budget.
Include varying aesthetic preferences: some may prefer minimalist design, others vibrant and bold styles.
Represent different levels of familiarity with technology brands.
"""
factory = TinyPersonFactory(context_text=factory_context)

# Generate 5 diverse agents
print("Generating a panel of 5 diverse agents...")
people = factory.generate_people(number_of_people=5, verbose=True)
print(f"\nGenerated {len(people)} agents.")

### 6.2. Have Each Generated Agent Evaluate the Ads and Extract Choices

We'll loop through our newly generated agents, set their context, have them evaluate the ads, and then extract their choices.

In [ ]:
choices = []
if not people:
    print("No agents were generated, skipping evaluation.")
else:
    print("\n--- Starting Ad Evaluation with Generated Panel ---")
    for i, person in enumerate(people):
        print(f"\nProcessing agent {i+1}/{len(people)}: {person.name} ({person.get('occupation').get('title', 'N/A')})")
        person.change_context([situation])
        person.listen_and_act(eval_request_msg)
        
        print(f"Extracting choice from {person.name}...")
        # It's good practice to clear agent's memory if the same agent is used for multiple independent tasks,
        # but here each listen_and_act is a fresh sequence for this specific task.
        # person.episodic_memory.clear_all_interactions() # Optional: if you want to ensure extraction is only from the last interaction
        
        res = extractor.extract_results_from_agent(
            person=person, 
            extraction_objective=extraction_objective, 
            situation=situation,
            fields=["ad_number", "ad_title"],
            fields_hints={"ad_number": "Must be an integer (1, 2, or 3), not a string like 'AD 1'."},
            verbose=False # Set to True for debugging individual extractions
        )
        
        if res and res.get('ad_number') is not None:
            choices.append(res)
            print(f"{person.name} chose: AD {res.get('ad_number')} - {res.get('ad_title')}")
        else:
            print(f"Could not extract a definitive choice from {person.name}. Raw response: {person.episodic_memory.retrieve_recent()[-1]['content'] if person.episodic_memory.retrieve_recent() else 'No interaction'}")
        print("---------------------")

### 6.3. Tally the Votes and Determine the Winning Ad

In [ ]:
print("\n--- Aggregated Ad Preferences ---")
votes = {}
ad_titles = { # Store titles for prettier output
    1: "LG 4K Ultra HD TV (AD 1)",
    2: "Samsung TV Lineup (AD 2)",
    3: "Wayfair 55 Inch Tv (AD 3)"
}

if not choices:
    print("No choices were extracted from the generated agents.")
else:
    for choice in choices:
        try:
            # Ensure ad_number is an integer before using it as a dict key
            ad_number = int(choice.get('ad_number', -1)) # Default to -1 if not found or not int
            if ad_number not in ad_titles:
                print(f"Warning: Extracted unknown ad_number '{ad_number}' from choice: {choice}")
                continue
        except ValueError:
            print(f"Warning: Could not convert ad_number to int from choice: {choice}")
            continue
            
        if ad_number not in votes:
            votes[ad_number] = 0
        votes[ad_number] += 1

    print("\nVotes per Ad:")
    for ad_num, count in votes.items():
        print(f"{ad_titles.get(ad_num, f'AD {ad_num}')}: {count} vote(s)")

    if votes:
        # Determine the winning ad
        winner_ad_number = max(votes, key=votes.get)
        print(f"\nThe winning ad is: {ad_titles.get(winner_ad_number, f'AD {winner_ad_number}')} with {votes[winner_ad_number]} vote(s).")
    else:
        print("\nNo valid votes were cast to determine a winner.")

## Conclusion

This notebook demonstrated how TinyTroupe can be used to simulate an audience's reaction to different advertisement copies. By generating diverse agent personas and tasking them with evaluating the ads, we can gather qualitative feedback (their reasoning) and quantitative data (which ad was chosen most often).

This approach can be extended to:
- Test different marketing messages.
- Evaluate product descriptions or names.
- Understand how different demographic groups might perceive content.
- Generate synthetic survey data for initial insights before conducting expensive real-world surveys.